<a href="https://colab.research.google.com/github/hajyhia/Airbnb_Berlin_Price_predic/blob/main/04_Airbnb_Berlin_Feature_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install missingno
# !pip install geopy

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
# from pandas_profiling import ProfileReport
import numpy as np
import missingno as msno
sns.set()
# plt.style.use('ggplot')
plt.style.use('seaborn-v0_8')
import warnings
import datetime as dt

from sklearn import ensemble, tree, linear_model
from sklearn import tree
from sklearn.metrics import accuracy_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer

from scipy.stats import pearsonr
from scipy.stats import ks_2samp
from scipy.stats import norm
from scipy import stats
from scipy.stats import chisquare
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway

# Ignore warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df_Feature_Engineering = pd.read_pickle('/content/drive/My Drive/Airbnb/df_Feature_Engineering.pkl')

# Feature Selection

In [5]:
df_Feature_Selection = pd.read_pickle('/content/drive/My Drive/Airbnb/df_Feature_Selection.pkl')
df_Feature_Selection.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23460 entries, 0 to 23459
Data columns (total 34 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Accuracy Rating               23460 non-null  float64       
 1   Bathrooms                     23460 non-null  float64       
 2   Checkin Rating                23460 non-null  float64       
 3   Cleanliness Rating            23460 non-null  float64       
 4   Communication Rating          23460 non-null  float64       
 5   Latitude                      23460 non-null  float64       
 6   Location Rating               23460 non-null  float64       
 7   Longitude                     23460 non-null  float64       
 8   Overall Rating                23460 non-null  float64       
 9   Price                         23460 non-null  float64       
 10  Value Rating                  23460 non-null  float64       
 11  Comments                    

In [6]:
df_Feature_Selection['Distance From Center Reduced']

,Distance From Center Reduced
0,km_4
1,km_2
2,km_4
3,km_2
4,km_4
...,...
23455,km_25
23456,km_25
23457,km_4
23458,km_25


In [7]:
df = df_Feature_Selection.copy()

In [8]:
df = df.drop(columns=['Host Since','Host Since Year', 'Postal Code', 'Property Type', 'Comments','neighbourhood','Distance From Center Reduced', 'Longitude', 'Latitude']
             , inplace=False)

## One-Hot Enciding and Label Encoding

In [9]:
from sklearn.preprocessing import OrdinalEncoder

df_object =  df.select_dtypes(include = ['object','category']).columns
for col in df_object:
  ord_enc = OrdinalEncoder()
  df[[col]] = ord_enc.fit_transform(df[[col]]).astype('int')

In [10]:
df.head(2)

,Accuracy Rating,Bathrooms,Checkin Rating,Cleanliness Rating,Communication Rating,Location Rating,Overall Rating,Price,Value Rating,Host Response Time,Is Superhost,Neighborhood Group,Is Exact Location,Room Type,Instant Bookable,Accomodates,Bedrooms,Beds,Guests Included,Min Nights,Reviews,Property Type Reduced,Postal Code Reduced,Distance From Center,Host Since From Now
0,10.0,1.0,10.0,10.0,10.0,9.0,100.0,17.0,10.0,2,False,6,True,1,False,2.0,1.0,1.0,1.0,2.000000,7.0,0,0,5.1,17
1,9.0,1.0,9.0,9.0,9.0,10.0,92.0,90.0,9.0,2,False,6,True,0,False,4.0,1.0,2.0,2.0,1.727273,144.0,0,0,3.7,17


In [11]:
# Creating Variables dataframeS
varSel = pd.DataFrame({'Variable': df.columns.drop('Price')})
varSel

,Variable
0,Accuracy Rating
1,Bathrooms
2,Checkin Rating
3,Cleanliness Rating
4,Communication Rating
5,Location Rating
6,Overall Rating
7,Value Rating
8,Host Response Time
9,Is Superhost


In [12]:
df.to_csv('/content/drive/My Drive/Airbnb/df_Feature_Selection.csv')

In [13]:
nm = df.columns.drop('Price')
nm = nm.append(pd.Index(['Price']))
nm

Index(['Accuracy Rating', 'Bathrooms', 'Checkin Rating', 'Cleanliness Rating',
       'Communication Rating', 'Location Rating', 'Overall Rating',
       'Value Rating', 'Host Response Time', 'Is Superhost',
       'Neighborhood Group', 'Is Exact Location', 'Room Type',
       'Instant Bookable', 'Accomodates', 'Bedrooms', 'Beds',
       'Guests Included', 'Min Nights', 'Reviews', 'Property Type Reduced',
       'Postal Code Reduced', 'Distance From Center', 'Host Since From Now',
       'Price'],
      dtype='object')

In [14]:
X = df.drop(columns='Price', inplace=False)
y = df['Price']

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVR
from sklearn.linear_model import Ridge


In [16]:
varSel.to_csv('/content/drive/My Drive/Airbnb/Feature_Selection_models.csv')

In [17]:
varSel.shape

(24, 1)

## Multivariable Analysis

In [18]:
# Fit models and determine if a feature is selected (1) or not (0)
lasso = Lasso(alpha=5).fit(X, y)
lasso_selected = (np.abs(lasso.coef_) > 0).astype(int)

# Fit Ridge model
ridge = Ridge(alpha=5).fit(X, y)
ridge_selected = (np.abs(ridge.coef_) > 0).astype(int)

gb = GradientBoostingRegressor().fit(X, y)
gb_selected = (gb.feature_importances_ > 0).astype(int)

rf = RandomForestRegressor().fit(X, y)
rf_selected = (rf.feature_importances_ > 0).astype(int)

# Create a DataFrame to store results
selection_df = pd.DataFrame({
    'Feature': X.columns,
    'Lasso': lasso_selected,
    'GradientBoost': gb_selected,
    'RandomForest': rf_selected,
    'Ridge': ridge_selected
})

# Sum the number of selections for each feature
selection_df['Sum'] = selection_df[['Lasso', 'GradientBoost', 'RandomForest','Ridge']].sum(axis=1)

# Output the results
selection_df

,Feature,Lasso,GradientBoost,RandomForest,Ridge,Sum
0,Accuracy Rating,0,1,1,1,3
1,Bathrooms,0,0,0,0,0
2,Checkin Rating,0,1,1,1,3
3,Cleanliness Rating,0,1,1,1,3
4,Communication Rating,0,1,1,1,3
5,Location Rating,0,1,1,1,3
6,Overall Rating,1,1,1,1,4
7,Value Rating,0,1,1,1,3
8,Host Response Time,1,1,1,1,4
9,Is Superhost,0,1,1,1,3


In [19]:
selection_df.to_csv('/content/selection_variable_modele.csv')

In [20]:
selection_df.shape

(24, 6)

In [21]:
selection_df[['Feature', 'Sum']].sort_values(by='Sum', ascending=False)

,Feature,Sum
6,Overall Rating,4
17,Guests Included,4
19,Reviews,4
20,Property Type Reduced,4
12,Room Type,4
14,Accomodates,4
15,Bedrooms,4
8,Host Response Time,4
22,Distance From Center,4
16,Beds,4


In [22]:
selection_df[selection_df['Sum'] >= 3].shape

(21, 6)

In [23]:
#Selecting variables with a sum of selections >= 3
selected_variables = selection_df[selection_df['Sum'] >= 3]['Feature']
selected_variables.shape

(21,)

In [24]:
selected_variables

,Feature
0,Accuracy Rating
2,Checkin Rating
3,Cleanliness Rating
4,Communication Rating
5,Location Rating
6,Overall Rating
7,Value Rating
8,Host Response Time
9,Is Superhost
10,Neighborhood Group


## Creating DataFrame with most valuable variables

In [25]:
df_model = df.loc[:,selected_variables]
df_model['Price'] = df['Price'].copy()

# Output the result to verify
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23460 entries, 0 to 23459
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Accuracy Rating        23460 non-null  float64
 1   Checkin Rating         23460 non-null  float64
 2   Cleanliness Rating     23460 non-null  float64
 3   Communication Rating   23460 non-null  float64
 4   Location Rating        23460 non-null  float64
 5   Overall Rating         23460 non-null  float64
 6   Value Rating           23460 non-null  float64
 7   Host Response Time     23460 non-null  int64  
 8   Is Superhost           23460 non-null  bool   
 9   Neighborhood Group     23460 non-null  int64  
 10  Room Type              23460 non-null  int64  
 11  Accomodates            23460 non-null  float64
 12  Bedrooms               23460 non-null  float64
 13  Beds                   23460 non-null  float64
 14  Guests Included        23460 non-null  float64
 15  Mi

## Store state

In [26]:
selection_df.to_pickle("/content/drive/My Drive/Airbnb/feature_selection_df.pkl")
df_model.to_pickle("/content/drive/My Drive/Airbnb/df_Model_Selection.pkl")